# Project Overview

**Important: This project requires GPUs. Before you start, please go to "Runtime", then click on "Change runtime type", and afterwards choose either a T4, L4, or A100 GPU.**

Throughout this course you will build an AI-powered retail business with three components:

1. **Intelligent Marketing Agent (Project 1 — this module)**  
   Send personalised emails to each customer. Hook quality determines how likely a customer is to visit the shop.  
   *Metric: number of website visits per day.*

2. **AI-Powered Recommendation System (Project 2)**  
   Recommend relevant items to customers who visit the shop.  
   *Metric: total order revenue per day.*

3. **Reinforcement Learning Delivery Optimisation (Project 3)**  
   Optimise delivery routes (CVRP-TW) to minimise fleet costs while respecting delivery time windows.  
   *Metric: total dispatch cost.*

To track everything, a SQLite database stores customers, items, shop visits, and orders. Customer behaviour is simulated by a **World Simulator**, which acts as a proxy for the real world.


# Project 1 — Intelligent Marketing Agent

You will build an Intelligent Marketing Agent (IMA) that sends personalised promotional emails to customers.

## Customer Segments

Each customer belongs to one of five segments based on purchase history and average order value (AOV):

| Segment | Code | Description | Marketing Goal |
|---------|------|-------------|----------------|
| Newcomers | N | No purchases yet | Build trust, encourage first purchase |
| High-Value | H | High AOV, frequent buyer | Reinforce loyalty, reward spending |
| Regular | R | Moderate AOV, steady buyer | Nurture relationship, highlight new arrivals |
| Low-Value | L | Low AOV, price-sensitive | Increase AOV via bundles or free-shipping thresholds |
| At-Risk | A | Previously active, now inactive | Urgent re-engagement with a compelling return offer |

For each segment there exists a hidden optimal hook $h^*$ that maximises visit probability. Your goal is to discover it.

## Email Format

Each email is $e = (h, r)$: a marketing hook $h$ followed by product recommendations $r$. In Project 1 the recommendations are random, so only the hook quality matters.

## Hook Quality Score

Hook quality is measured by semantic similarity to the optimal hook, using BERT embeddings and cosine similarity:

$$
\texttt{cos-sim}(v, w) := \begin{cases} \dfrac{\langle v, w \rangle}{\|v\|\,\|w\|} & \text{if } v \neq 0 \neq w \\ 0 & \text{otherwise} \end{cases}
$$

For a customer with optimal hook $h^*$, the quality of a generated hook $h$ is:

$$\text{quality}(h) = \texttt{cos-sim}\bigl(\text{BERT}(h),\, \text{BERT}(h^*)\bigr)$$

This score updates the customer's visit probability: the higher the quality, the more likely they visit.

## Finding the Optimal Hook — OPRO

We frame hook discovery as an optimisation problem:

$$\hat{h} := \arg\max_{h}\; \texttt{cos-sim}\bigl(\text{BERT}(h),\, \text{BERT}(h^*)\bigr)$$

We solve this with **OPRO (Optimisation by PROmpting)** ([paper](https://arxiv.org/abs/2309.03409)): an LLM iteratively proposes improved hooks based on the performance of previous ones.

**Cycle (5 days):**
1. Use the current hook for all customers in a segment for 5 days.
2. Score it as `visits / (customers × days)`.
3. Feed the top-$k$ (hook, score) pairs to the LLM and ask it to generate a better hook.
4. Repeat.

In [ ]:
#@title ⚙️ Setup: install required packages and download project data
import os

repo_url = "https://github.com/eth-ainit-fs26/project.git"
branch = "week1"

if os.path.exists("project"):
    !git -C project pull origin {branch}
else:
    !git clone --branch {branch} --single-branch {repo_url}

# Move into the project folder so all imports resolve correctly
if os.path.basename(os.getcwd()) != "Project1":
    os.chdir("project/Project1")

# Install dependencies
%pip install -r requirements.txt -q

# Import required modules
from app.world_sim_p1 import WorldSimulatorP1, LoggingConfig, MarketingSession
from services.llm_service import LLMService
from system_config.system_parameters import SystemParameters
from agents.hooks import HookEvaluator
import numpy as np
import pandas as pd
import ast
import os
import sys
from typing import Tuple

print("✅ All modules imported successfully")

## 1. Database Setup

Loads customers, products, and optimal hooks into the database. No implementation required.


In [ ]:
from models.data_loader import load_data_from_files, setup_real_data, setup_real_transactions, create_system_parameters_from_data
from models.database import reset_database, initialize_database
from peewee import IntegrityError
import numpy as np

np.random.seed(42)

print("Loading data from TSV files...")
customers_df, products_df, hooks_df = load_data_from_files()
customers_df = customers_df.head(40)
print(f"Loaded {len(customers_df)} customers and {len(products_df)} products")

print("Resetting and initialising database...")
reset_database()
initialize_database()

print("Seeding database with customers and products...")
try:
    setup_real_data(customers_df, products_df)
except IntegrityError:
    print("Data already present, continuing...")
setup_real_transactions(customers_df, products_df)
print("Dataset setup complete")

## 2. Configure LLM Service

Initialises the local Qwen 2.5 model used for hook generation and optimisation. No implementation required.


In [ ]:
from huggingface_hub import hf_hub_download
from services.llm_service import LlamaFileProvider

model_path = hf_hub_download(
    repo_id="Qwen/Qwen2.5-1.5B-Instruct-GGUF",
    filename="qwen2.5-1.5b-instruct-q5_k_m.gguf",
)

llm_provider = LlamaFileProvider(model_path=model_path, n_ctx=8192, verbose=False)
llm_service = LLMService(provider=llm_provider)

print("LLM service ready")

## 3. System Parameters & Logging

Sets up simulation constants and logging configuration. No implementation required.


In [ ]:
from models.data_loader import build_hook_evaluator

system_parameters, hook_evaluator = build_hook_evaluator(hooks_df, customers_df, products_df)

# Configure logging — set flags to True to see more output while debugging
logging_config = LoggingConfig(
    daily_progress=False,
    marketing_hooks=False,
    marketing_individual=False,
    shopping_phase=False,
    accuracy_daily=False,
    accuracy_cycle=False,
    opro_optimization=False,
    opro_prompts=False,
    opro_fallbacks=False,
    day_advancement=False,
    final_history=False,
    setup_messages=False,
)

## 4. Your Implementation

### 4.1 HookEvaluator

The `HookEvaluator` scores a generated hook against the optimal hook for a given segment using BERT cosine similarity.

**Implement** the two methods marked `raise NotImplementedError`:

- `evaluate_hook_quality_with_embedding(hook_embedding, segment)` — compute cosine similarity between a pre-computed hook embedding and the stored optimal embedding for the segment.

Useful:
- `np` — NumPy is available

In [ ]:
def custom_cosine_similarity(self, vec1, vec2):
    """
    Custom cosine similarity implementation.

    Args:
        vec1: First vector (numpy array)
        vec2: Second vector (numpy array)

    Returns:
        Similarity score between 0.0 and 1.0

    Ideas to try:
    - Standard cosine similarity: dot(normalized_vec1, normalized_vec2)
    - Euclidean distance converted to similarity: 1.0 / (1.0 + distance)
    - Manhattan distance: 1.0 / (1.0 + manhattan_distance)

    Hint: Use np.linalg.norm() for vector normalization
    Hint: Use np.dot() for dot product
    """
    # vec1 and vec2 are both numpy arrays (vectors of floats).
    # Cosine similarity formula:  dot(v1, v2) / (|v1| * |v2|)
    # Useful functions:
    #    np.dot(vec1, vec2)       — dot product of two vectors
    #    np.linalg.norm(vec)      — length (magnitude) of a vector
    # 🎯🎯🎯 Target 🎯🎯🎯

    # Calculate cosine similarity of normalized vectors
    # 🎯🎯🎯 Target 🎯🎯🎯

    # Ensure result is in [0, 1] range and return it
    # 🎯🎯🎯 Target 🎯🎯🎯
    raise NotImplementedError("Implement HookEvaluator.custom_cosine_similarity")


def custom_evaluate_hook_quality_with_embedding(self, hook_embedding, customer_segment):
    """
    Custom hook quality evaluation using pre-computed embedding.

    Args:
        hook_embedding: Pre-computed normalized embedding of the hook (numpy array)
        customer_segment: Customer segment (N, H, A, L, R, or Default)

    Returns:
        Hook quality score between 0.0 and 1.0

    Available data:
    - self._optimal_embeddings: Dict mapping segments to optimal hook embeddings
    - self.cosine_similarity(): Your custom similarity function

    Ideas to try:
    - Compare hook_embedding with optimal embedding for the segment
    - Apply segment-specific weighting factors
    - Combine multiple similarity metrics
    - Add bonus/penalty based on segment characteristics

    Hint: Get optimal embedding with self._optimal_embeddings.get(customer_segment, self._optimal_embeddings["Default"])
    Hint: Use self.cosine_similarity(hook_embedding, optimal_embedding)
    """
    try:
        # Get pre-computed optimal hook embedding
        optimal_embedding = self._optimal_embeddings.get(
            customer_segment,
            self._optimal_embeddings["Default"]
        )

        # Calculate cosine similarity between hook and optimal embeddings by using self.cosine_similarity
        # 🎯🎯🎯 Target 🎯🎯🎯

        # Return the result as a float type value
        # 🎯🎯🎯 Target 🎯🎯🎯
        raise NotImplementedError("Implement HookEvaluator.custom_evaluate_hook_quality_with_embedding")

    except Exception as e:
        print(f"Error evaluating hook quality with pre-computed embedding: {e}")
        return 0.0


from app.patching import patch_hook_evaluator

patch_hook_evaluator(custom_cosine_similarity, custom_evaluate_hook_quality_with_embedding)

### 4.2 MarketingSession

`MarketingSession` runs the per-customer marketing workflow. Two methods drive it:

**`run_session()`** — called once per customer per day:
1. Determine the customer's segment (`classify_customer_segment`)
2. Select the current hook for that segment
3. Call `run_step(...)` to execute the marketing interaction
4. Update the customer's visit probability based on hook quality
5. Store the result in `self.session_history` and return it

**`run_step(session_history, hook, segment, llm_instruction)`** — the core interaction:
1. Generate a `MarketingEmail` via `self.marketing_agent.createMessage`
2. Simulate a customer response via `self.customer_response.create_response`
3. Score the hook quality

**Your task:** plug in your `HookEvaluator` to compute `hook_quality_score` inside `run_step`.


In [ ]:
def run_session(self):
    """Run the complete marketing session."""
    # 1. Classification of customer segment
    customer_segment = self.classify_customer_segment()  # CustomerSegment.HIGH_VALUE_CUSTOMERS.value

    # 3. Use pre-generated hook from WorldSimulator
    selected_hook = self.current_hooks.get(customer_segment)

    if not selected_hook:
        # Fallback to optimal hooks if current_hooks is empty
        selected_hook = self.optimal_hooks.get(customer_segment,
                                               self.optimal_hooks.get("Default", "Special offers just for you!"))

    marketing_email, client_response, hook_quality_score = self.run_step(self.session_history, selected_hook,
                                                                         customer_segment,
                                                                         self.llm_instruction_prompt.get(
                                                                             customer_segment))

    # Update visit probability
    self.update_customer_visit_proba(hook_quality_score)

    self.session_history.append({
        'marketing_email': marketing_email,
        'client_response': client_response,
        'customer_segment': customer_segment,
        'email_hook': selected_hook,
        'llm_hook_instruction': self.llm_instruction_prompt.get(customer_segment)
    })

    return self.session_history


def run_step(self, session_history, email_hook=None, customer_segment=None, llm_hook_instruction=None):
    """Run a single step of the marketing session."""
    # Marketing agent creates MarketingEmail for client
    # Pass the email hook, segment, and LLM instruction to potentially customize the message
    marketing_email = self.marketing_agent.createMessage(self.client, email_hook, customer_segment,
                                                         llm_hook_instruction)

    # Client creates response based on the marketing email object
    client_response = self.customer_response.create_response(marketing_email)

    # Use the hook evaluator to get the score of the hook
    hook_quality_score = 0.5  # Default score
    if hasattr(marketing_email, 'marketing_hook') and marketing_email.marketing_hook:

        # Replace hook_quality_score with the real score from your evaluator.
        # Make use of self.evaluate_hook_quality method
        # 🎯🎯🎯 Target 🎯🎯🎯
        raise NotImplementedError("Get hook quality score using your custom evaluator")

    return marketing_email, client_response, hook_quality_score


from app.patching import patch_marketing_session

patch_marketing_session(run_session, run_step)

### 4.3 Initialise WorldSimulatorP1

Creates the `WorldSimulatorP1` instance used for the rest of the notebook. No implementation required.


In [ ]:
from app.patching import confirm_world_simulator

use_opro2_mode = True  # False → original OPRO (optimises prompts); True → OPRO2 (optimises hooks)

w = WorldSimulatorP1(
    llm_service=llm_service,
    system_parameters=system_parameters,
    hook_evaluator=hook_evaluator,
    simulation_duration=50,
    opro_cycle_days=5,
    logging_config=logging_config,
    use_opro2=use_opro2_mode,
)
confirm_world_simulator(w, use_opro2_mode)

### 4.4 OPRO Cycle Accuracy

**Implement** `calculate_segment_hook_accuracy`: given the visit outcomes over one 5-day OPRO cycle, return the visit rate (visits / total customers) for each segment.

This accuracy feeds directly into the OPRO meta-prompt as the performance score for each hook.


In [ ]:
from typing import Optional, Dict


def calculate_segment_hook_accuracy(self, day_number=None):
    """
    Calculate hook accuracy for each customer segment based on shop visit rates.

    Accuracy is defined as: (number of customers in segment who visited) / (total customers in segment)

    Args:
        day_number: Simulation day (defaults to current day)

    Returns:
        Dict[str, float]: Dictionary mapping segment to accuracy score (0.0-1.0)

    Available data:
    - self.customer_registry: Access to all customer data with segments
    - self.shop_component: Access to shop visit data for any day
    - self.current_simulation_day: Current simulation day if day_number is None

    Algorithm:
    1. Get all customers and group them by segment
    2. Count total customers per segment
    3. Get shop visits for the specified day
    4. Count how many customers from each segment visited
    5. Calculate accuracy = visited_count / total_count per segment

    Hint: Use self.customer_registry.get_all_customers()
    Hint: Use self.shop_component.get_visits_for_day(day_number)
    Hint: Create a dictionary to track {'total': count, 'visited': count} per segment
    Hint: Return a dict mapping segment -> accuracy (0.0-1.0)
    """
    if day_number is None:
        day_number = self.current_simulation_day

    # Get all customers by using the self.customer_registry.get_all_customers method
    # 🎯🎯🎯 Target 🎯🎯🎯

    # Count total customers per segment

    # You need a data structure that maps each segment code (e.g. 'N', 'H')
    # to two numbers: how many customers are in that segment in total, and
    # how many of them visited. Start by filling the totals here — you'll
    # fill in the visit counts in Step 4.
    # 🎯🎯🎯 Target 🎯🎯🎯

    # Get all shop visits for the day

    # Make use of the self.shop_component.get_visits_for_day method by providing it with the number of day.
    # After retrieving the day visits, extract the customer ids (cid) that did visit on the specific day.
    # 🎯🎯🎯 Target 🎯🎯🎯

    # Count customers who visited per segment

    # For every customer, find if the visited on current day, their
    # respective segment and increment that segment's visit counter.
    # 🎯🎯🎯 Target 🎯🎯🎯

    # Compute the accuracy score for each segment and return the result.

    # Accuracy = (customers who visited) / (total customers in segment).
    # Make sure you handle the edge case where a segment has zero customers
    # to avoid a division-by-zero error.
    # Return a plain dict mapping each segment code to its float score.
    # 🎯🎯🎯 Target 🎯🎯🎯

    raise NotImplementedError("WorldSimulatorP1.calculate_segment_hook_accuracy")


from app.patching import patch_world_simulator

patch_world_simulator(calculate_segment_hook_accuracy)

### 4.5 OPRO Meta-Prompt

**Implement** `create_meta_prompt` in the `Opro2` subclass: write a prompt template that presents the LLM with past (hook, score) pairs and asks it to generate a better hook for the given segment.

See `create_meta_prompt_example` for a reference implementation.


In [ ]:
from dataclasses import dataclass
from models.models import CustomerSegment
from agents.opro import OptimizerEntry2, Opro2
from typing import List

# Feel free to redefine segment_descriptions here if needed
segment_descriptions = {
    CustomerSegment.NEWCOMERS.value: "new customers who have never purchased before",
    CustomerSegment.HIGH_VALUE_CUSTOMERS.value: "customers who make frequent large purchases",
    CustomerSegment.REGULAR_CUSTOMERS.value: "regular customers who occasionally make purchases",
    CustomerSegment.AT_RISK_CUSTOMERS.value: "regular customers who haven't purchased recently or show low satisfaction",
    CustomerSegment.LOW_VALUE_CUSTOMERS.value: "customers who make purchases with a low average value"
}


@dataclass
class OptimizerEntry2:
    """
    Data structure for OPRO optimization entries containing performance metrics
    and associated hooks for a specific customer segment.

    accuracy_score represents the actual visit rate accuracy for the cycle
    (range: 0.0 to 1.0)
    """
    accuracy_score: float  # Visit rate accuracy score (0.0 to 1.0): proportion of customers who visited
    hook: str  # Marketing hook text that achieved this accuracy


def create_meta_prompt_example(self, customer_segment: str, top_performers: List[OptimizerEntry2]) -> str:
    segment_desc = segment_descriptions.get(customer_segment, "general customers")
    # Sort best-first
    top_performers = sorted(top_performers, key=lambda e: e.accuracy_score, reverse=True)

    # Build examples block
    examples_text = []
    for i, entry in enumerate(top_performers, 1):
        examples_text.append(
            f"Example {i} — Visit Rate: {entry.accuracy_score:.3f}\n"
            f'Hook: "{entry.hook}"'
        )
    examples_block = "\n\n".join(examples_text)

    best = top_performers[0]

    # Create meta-prompt template + fill in details
    meta_prompt = f"""
You are a concise marketing copywriter.

TARGET AUDIENCE: {segment_desc}

GOAL: Make a tiny improvement to the BEST hook to increase visit rate.

BEST HOOK (baseline):
- Visit Rate: {best.accuracy_score:.3f}
- Text: "{best.hook}"

OTHER TOP HOOKS (for reference):
{examples_block}

RULES:
- Start from the BEST hook.
- Do NOT rewrite completely.
- Do NOT explain.

OUTPUT:
Write ONLY the improved hook (one line, no quotes, no extra text)."""

    return meta_prompt


def create_meta_prompt(self, customer_segment: str, top_performers: List[OptimizerEntry2]) -> str:
    # Copy the code from create_meta_prompt_example as a way of starting off with this part of the project.
    # Then, make educated changes and attempt to increase yours scores by changing the prompt.
    # There is no right or wrong answer here — the goal is to experiment with prompt design and see how it affects the optimisation process.
    # 🎯🎯🎯 Target 🎯🎯🎯
    raise NotImplementedError("Implement Opro2.create_meta_prompt")


from app.patching import patch_opro, show_meta_prompt_example

patch_opro(create_meta_prompt)
show_meta_prompt_example(create_meta_prompt)

## 5. Run the Simulation

### 5.1 Inspect Customer Data

Before running, review the loaded customers. Visit probabilities are initialised to `0.1`.


In [ ]:
from app.simulation_runner import display_customers

display_customers(w)

### 5.2 Run & Visualise

Runs the full marketing campaign with OPRO optimisation over the configured number of days. You can collapse this cell after running it.


In [ ]:
from app.simulation_runner import SimulationDashboard

dash = SimulationDashboard(w, use_opro2_mode)
dash.display()
dash.run()

## 6. Cleanup

Close the database connection to release resources.


In [ ]:
from app.simulation_runner import cleanup

cleanup()